In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.cluster import KMeans

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, mean_absolute_error, r2_score
)
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred):
        return mean_squared_error(y_true, y_pred) ** 0.5

from sklearn.metrics import silhouette_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.datasets import fetch_california_housing
from sklearn.pipeline import make_pipeline


In [ ]:
def charger_immobilier():
  
    data = fetch_california_housing()
    X = data.data
    y = data.target 

    print(f"California Housing : {X.shape[0]} lignes, {X.shape[1]} variables")
    print(f"Variables : {list(data.feature_names)}")
    print(f"Cible : prix médian en centaines de milliers de $")
    print(f"  min={y.min():.2f}, max={y.max():.2f}, moyenne={y.mean():.2f}")

    return X, y


def evaluer_regression(nom_modele, modele, X_train, X_test, y_train, y_test):
  
    modele.fit(X_train, y_train)
    y_pred = modele.predict(X_test)

    r2   = r2_score(y_test, y_pred)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)

    print(f"{nom_modele:<22} : R2={r2:.2f}  MAE={mae:.2f}  RMSE={rmse:.2f}")
    return {"r2": r2, "mae": mae, "rmse": rmse}


X_immo, y_immo = charger_immobilier()

X_tr_i, X_te_i, y_tr_i, y_te_i = train_test_split(
    X_immo, y_immo, test_size=0.2, random_state=42
)
scaler_i = StandardScaler()
X_tr_is = scaler_i.fit_transform(X_tr_i)
X_te_is = scaler_i.transform(X_te_i)

print("\n--- Résultats ---")
res_lr = evaluer_regression(
    "LinearRegression",
    LinearRegression(), X_tr_is, X_te_is, y_tr_i, y_te_i
)
res_rf = evaluer_regression(
    "RandomForest",
    RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    X_tr_is, X_te_is, y_tr_i, y_te_i
)


In [ ]:


data_immo = fetch_california_housing()
lr_interp = LinearRegression().fit(X_tr_is, y_tr_i)

print("Coefficients de la régression linéaire :")
print("(+coef = augmente le prix | -coef = baisse le prix)")
print()
for nom, poids in zip(data_immo.feature_names, lr_interp.coef_):
    print(f"  {nom:>15} : {poids:+.3f}")


In [ ]:

lr_small = LinearRegression()
lr_small.fit(X_tr_is[:100], y_tr_i[:100])
r2_small = r2_score(y_te_i, lr_small.predict(X_te_is))
print(f"R2 avec 100 lignes seulement : {r2_small:.3f}")
print(f"R2 avec le dataset complet   : {res_lr['r2']:.3f}")



quartier_fictif = np.zeros((1, X_immo.shape[1]))
quartier_fictif[0, 0] = 0.0  
quartier_fictif[0, 4] = 9000 

quartier_fictif_scaled = scaler_i.transform(quartier_fictif)
prix_predit = lr_interp.predict(quartier_fictif_scaled)[0]

print(f"Quartier fictif (revenu=0, pop=9000) → prix prédit : {prix_predit:.2f} (×100k$)")


In [ ]:
def charger_airbnb(url_csv):
  
    try:
        df = pd.read_csv(url_csv, low_memory=False)
    except Exception as e:
        print(f"⚠️  Impossible de charger depuis l'URL : {e}")
        print("→ On génère des données synthétiques pour la démonstration.")
        return None

    colonnes_utiles = [
        "price", "minimum_nights", "number_of_reviews", "availability_365"
    ]
    colonnes_dispo = [c for c in colonnes_utiles if c in df.columns]

    if not colonnes_dispo:
        return None

    df_clean = df[colonnes_dispo].copy()

    if "price" in df_clean.columns and df_clean["price"].dtype == object:
        df_clean["price"] = df_clean["price"].str.replace("[$,]", "", regex=True)
        df_clean["price"] = pd.to_numeric(df_clean["price"], errors="coerce")

    df_clean = df_clean.dropna()
    if "price" in df_clean.columns:
        df_clean = df_clean[df_clean["price"] > 0]
        df_clean = df_clean[df_clean["price"] < 1000]  

    print(f"✅ AirBnB chargé : {len(df_clean)} lignes, {len(colonnes_dispo)} colonnes retenues")
    return df_clean


def generer_airbnb_synthetique():
    
    np.random.seed(42)
    n = 600

    g0 = pd.DataFrame({
        "price": np.random.normal(30, 10, n//3),
        "minimum_nights": np.random.normal(5, 2, n//3),
        "number_of_reviews": np.random.normal(10, 5, n//3),
        "availability_365": np.random.normal(200, 50, n//3)
    })
    g1 = pd.DataFrame({
        "price": np.random.normal(80, 15, n//3),
        "minimum_nights": np.random.normal(2, 1, n//3),
        "number_of_reviews": np.random.normal(50, 15, n//3),
        "availability_365": np.random.normal(150, 40, n//3)
    })
   
    g2 = pd.DataFrame({
        "price": np.random.normal(200, 40, n//3),
        "minimum_nights": np.random.normal(1, 0.5, n//3),
        "number_of_reviews": np.random.normal(80, 20, n//3),
        "availability_365": np.random.normal(300, 60, n//3)
    })

    df = pd.concat([g0, g1, g2], ignore_index=True)
    df = df.clip(lower=0) 
    print(f"✅ Dataset synthétique AirBnB : {len(df)} annonces, 4 variables")
    return df



URL_AIRBNB = "http://data.insideairbnb.com/france/ile-de-france/paris/2023-12-12/visualisations/listings.csv"

df_airbnb = charger_airbnb(URL_AIRBNB)
if df_airbnb is None:
    df_airbnb = generer_airbnb_synthetique()

df_airbnb.describe()

In [ ]:
def choisir_k(X_scaled, k_range=range(2, 9)):
   
    print(f"{'k':>4} | {'Inertie':>10} | {'Silhouette':>10}")
    print("-" * 32)

    resultats = []
    for k in k_range:
        km = KMeans(n_clusters=k, n_init=10, random_state=42)
        km.fit(X_scaled)
        sil = silhouette_score(X_scaled, km.labels_)
        print(f"{k:>4} | {km.inertia_:>10.1f} | {sil:>10.3f}")
        resultats.append((k, km.inertia_, sil))

    meilleur = max(resultats, key=lambda x: x[2])
    print(f"\n→ Meilleur k selon silhouette : k={meilleur[0]} (silhouette={meilleur[2]:.3f})")
    return meilleur[0]


scaler_ab = StandardScaler()
X_airbnb = df_airbnb.values
X_airbnb_scaled = scaler_ab.fit_transform(X_airbnb)

print("=" * 55)
print("  CHOIX DU BON K (avec standardisation)")
print("=" * 55)
k_optimal = choisir_k(X_airbnb_scaled)

In [ ]:
km_final = KMeans(n_clusters=k_optimal, n_init=10, random_state=42)
km_final.fit(X_airbnb_scaled)
df_airbnb["segment"] = km_final.labels_

print("\n=" * 55)
print(f"  PROFIL DES {k_optimal} SEGMENTS")
print("=" * 55)
print(df_airbnb.groupby("segment").mean().round(1).to_string())
print()
print("Taille de chaque segment :")
print(df_airbnb["segment"].value_counts().sort_index().to_string())

In [ ]:
def charger_spam(url):

    try:
        df = pd.read_csv(url, sep="\t", header=None, names=["label", "message"],
                         encoding="latin-1")
        messages = df["message"].tolist()
        labels = (df["label"] == "spam").astype(int).tolist()
        print(f"✅ Spam chargé : {len(messages)} messages")
        print(f"   Normal : {labels.count(0)} | Spam : {labels.count(1)}")
        print(f"   → Déséquilibre : {labels.count(1)/len(labels):.1%} de spams")
        return messages, labels
    except Exception as e:
        print(f"⚠️  Chargement impossible : {e}")
        return None, None


def generer_spam_synthetique():
    """Génère un mini jeu spam/normal synthétique pour la démonstration."""
    messages = [
        "FREE entry win cash prize call now urgent",
        "Congratulations you won 1000 pounds claim free prize",
        "urgent reply to claim your free gift voucher",
        "Win cash prize free mobile phone call this number",
        "FREE ringtones txt to win cash now",
        "WINNER claim your free prize urgent reply",
        "free cash award claim prize mobile",
        "Hi how are you doing today",
        "Can we meet tomorrow for lunch",
        "Ok i will call you later tonight",
        "Did you see the game last night",
        "Please send me the report by friday",
        "Happy birthday hope you have a great day",
        "Running late will be there in 10 minutes",
        "Sounds good see you at 6pm",
        "Can you pick up some milk on your way home",
        "The meeting is rescheduled to next monday",
        "Thanks for your help yesterday really appreciate it",
        "Are you coming to the party this weekend",
        "Just finished the project will send it now",
        "Good morning hope you slept well",
        "Dinner was great thanks for cooking",
        "Call me when you get a chance",
        "The train is delayed by 20 minutes",
        "See you at the office tomorrow morning",
        "Do you want to grab coffee before the meeting",
        "I sent you the file check your email",
        "Let me know if you need anything",
    ]
    labels = [1]*7 + [0]*21
    print(f"✅ Dataset synthétique spam : {len(messages)} messages")
    print(f"   Normal : {labels.count(0)} | Spam : {labels.count(1)}")
    return messages, labels


URL_SPAM = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"
URL_SPAM_TSV = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"

messages_spam, labels_spam = charger_spam(URL_SPAM_TSV)
if messages_spam is None:
    messages_spam, labels_spam = generer_spam_synthetique()

In [ ]:
def vectoriser_textes(messages, vectorizer=None):
 
    if vectorizer is None:
        vectorizer = TfidfVectorizer(lowercase=True, stop_words="english")
        X = vectorizer.fit_transform(messages)
    else:
        X = vectorizer.transform(messages)
    return X, vectorizer


def evaluer_spam(nom_modele, modele, X_train, X_test, y_train, y_test):
    """
    Entraîne, prédit, affiche precision/recall/F1 pour la classe spam.
    """
    modele.fit(X_train, y_train)
    y_pred = modele.predict(X_test)

    print(f"\n--- {nom_modele} ---")
    print(classification_report(y_test, y_pred,
                                target_names=["normal", "spam"],
                                zero_division=0))
    f1 = f1_score(y_test, y_pred, pos_label=1, zero_division=0)
    return f1


msg_train, msg_test, y_tr_sp, y_te_sp = train_test_split(
    messages_spam, labels_spam,
    test_size=0.2, random_state=42, stratify=labels_spam
)

X_tr_sp, vec_spam = vectoriser_textes(msg_train)
X_te_sp, _       = vectoriser_textes(msg_test, vectorizer=vec_spam)

print(f"Taille matrice train : {X_tr_sp.shape} (lignes × mots)")
print()

f1_nb = evaluer_spam("Naive Bayes", MultinomialNB(),
                     X_tr_sp, X_te_sp, y_tr_sp, y_te_sp)

f1_lr = evaluer_spam("Logistic Regression", LogisticRegression(max_iter=1000),
                     X_tr_sp, X_te_sp, y_tr_sp, y_te_sp)

In [ ]:

print("--- Edge case : message vide ---")
try:
    msg_vide = [""]
    X_vide, _ = vectoriser_textes(msg_vide, vectorizer=vec_spam)
    nb_model = MultinomialNB().fit(X_tr_sp, y_tr_sp)
    pred_vide = nb_model.predict(X_vide)[0]
    proba_vide = nb_model.predict_proba(X_vide)[0][1]
    print(f"Message vide → prédiction : {pred_vide} (proba spam : {proba_vide:.3f})")
    print("→ Pas de plantage. Le vectorizer renvoie un vecteur de zéros.")
except Exception as e:
    print(f"Erreur sur message vide : {e}")

print()

print("--- Adversarial : spam déguisé ---")
spam_deguise = ["salut ton colis t attend confirme ici gratuit"]
X_deguise, _ = vectoriser_textes(spam_deguise, vectorizer=vec_spam)
pred_d = nb_model.predict(X_deguise)[0]
proba_d = nb_model.predict_proba(X_deguise)[0][1]
print(f"Message : '{spam_deguise[0]}'")
print(f"→ Prédiction : {'SPAM' if pred_d==1 else 'normal'} (proba spam : {proba_d:.3f})")
print()


In [ ]:
def charger_sonar(url):
  
    try:
        df = pd.read_csv(url, header=None)
        X = df.iloc[:, :-1].values.astype(float)
        y = (df.iloc[:, -1] == "M").astype(int).values 

        nb_mines   = y.sum()
        nb_rochers = (y == 0).sum()
        print(f"✅ Sonar chargé : {X.shape[0]} lignes, {X.shape[1]} variables")
        print(f"   Mines : {nb_mines} | Rochers : {nb_rochers}")
        print(f"   → Classes quasi équilibrées ({nb_mines/len(y):.1%} de mines)")
        return X, y
    except Exception as e:
        return None, None


URL_SONAR = "https://archive.ics.uci.edu/ml/machine-learning-databases/undocumented/connectionist-bench/sonar/sonar.all-data"

X_sonar, y_sonar = charger_sonar(URL_SONAR)

if X_sonar is None:
    print("→ Génération de données synthétiques sonar...")
    np.random.seed(42)
    X_sonar = np.random.rand(208, 60)
    y_sonar = np.random.randint(0, 2, 208)


In [ ]:

X_tr_so, X_te_so, y_tr_so, y_te_so = train_test_split(
    X_sonar, y_sonar, test_size=0.2, random_state=42, stratify=y_sonar
)

scaler_so = StandardScaler()
X_tr_so_s = scaler_so.fit_transform(X_tr_so)
X_te_so_s = scaler_so.transform(X_te_so)

modeles_sonar = {
    "LogisticRegression" : LogisticRegression(max_iter=5000, random_state=42),
    "SVC (rbf)"          : SVC(kernel="rbf", random_state=42),
    "RandomForest"       : RandomForestClassifier(n_estimators=200, random_state=42),
}

print("=" * 50)
print("  RÉSULTATS SUR LE SONAR (avec standardisation)")
print("=" * 50)

for nom, m in modeles_sonar.items():
    m.fit(X_tr_so_s, y_tr_so)
    acc = accuracy_score(y_te_so, m.predict(X_te_so_s))
    f1  = f1_score(y_te_so, m.predict(X_te_so_s), zero_division=0)
    print(f"{nom:<22} : accuracy={acc:.2f}  F1={f1:.2f}")

print("\n→ Le SVM rbf est compétitif sur ce terrain : peu de lignes, beaucoup de variables.")

In [ ]:

for nom, m in modeles_sonar.items():
    m2_class = type(m) 
    if hasattr(m, 'get_params'):
        m2 = m.__class__(**m.get_params())
    else:
        m2 = m.__class__()
    m2.fit(X_tr_so, y_tr_so)  
    acc_brut = accuracy_score(y_te_so, m2.predict(X_te_so))
    print(f"{nom:<22} sans scale : accuracy={acc_brut:.2f}")


echo_panne = np.zeros((1, 60))
echo_panne_scaled = scaler_so.transform(echo_panne)

svc_model = modeles_sonar["SVC (rbf)"]
pred_panne = svc_model.predict(echo_panne_scaled)[0]
print(f"60 valeurs à zéro → prédiction : {'MINE' if pred_panne==1 else 'ROCHER'}")
